# Run: calibrate engine strength settings (~1700 Elo)

**Milestone:** "Rating-targeted play at around 1700 Elo" (README Teaching). Stockfish's `UCI_Elo`
is calibrated against other engines at a specific time control, not against humans on a physical
board with our think time. This notebook measures what our settings actually produce.

Method (proposed):
1. Play matches between Gambit's settings (`UCI_Elo` / `Skill Level`, `think_time_s`) and reference
   opponents of known strength (Stockfish at fixed `UCI_Elo`, or Maia-style human-like engines).
2. Estimate the Elo difference from the score: `diff = -400 * log10(1/score - 1)`.
3. Also measure ACPL of Gambit's own moves (habits.ipynb) to compare with human ACPL at 1700.

**TODO checklist**
- [ ] Choose reference opponents and time control.
- [ ] Run enough games for a useful interval (score SE ~ 0.5/sqrt(n)).
- [ ] Decide what to use below `min_elo` (Skill Level mapping) and update game.yaml.

In [ ]:
%run ./engine.ipynb

In [ ]:
import math
import chess.pgn
import pandas as pd

TARGETS = [1400, 1550, 1700, 1850, 2000]
GAMES_PER_SETTING = 20
assert find_engine_binary(), "install Stockfish first"


def elo_diff_from_score(score: float) -> float:
    score = min(max(score, 1e-3), 1 - 1e-3)
    return -400 * math.log10(1 / score - 1)


def play_match(white: Engine, black: Engine, max_plies: int = 300) -> str:
    board = chess.Board()
    while not board.is_game_over(claim_draw=True) and board.ply() < max_plies:
        board.push((white if board.turn == chess.WHITE else black).play(board))
    return board.result(claim_draw=True)

## Matches (TODO: pick the reference opponent)

In [ ]:
rows = []
for target in TARGETS:
    with Engine(elo=target) as gambit, Engine(elo=ENGINE_CFG["default_elo"]) as ref:   # TODO: real reference
        score = 0.0
        for i in range(GAMES_PER_SETTING):
            w, b = (gambit, ref) if i % 2 == 0 else (ref, gambit)
            r = play_match(w, b)
            pts = {"1-0": 1.0, "0-1": 0.0}.get(r, 0.5)
            score += pts if i % 2 == 0 else 1 - pts
        score /= GAMES_PER_SETTING
        rows.append({"target": target, "score_vs_ref": score, "elo_diff": elo_diff_from_score(score)})
pd.DataFrame(rows)